## Stage 2: Data Collection & Data Understanding

Understanding the dataset structure, class distribution, and image properties
before any modelling begins.

In [ ]:
!pip install -q kaggle

import os, cv2, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("✅ Imports done")

✅ Imports done


### 2.1 Dataset Download

In [ ]:
import os
from google.colab import files
files.upload()  # upload kaggle.json

os.makedirs('/root/.kaggle', exist_ok=True)
os.system('cp kaggle.json /root/.kaggle/kaggle.json')
os.system('chmod 600 /root/.kaggle/kaggle.json')
!kaggle --version

Saving kaggle.json to kaggle (6).json
Kaggle CLI 2.0.2


In [ ]:
!kaggle datasets download -d tanlikesmath/diabetic-retinopathy-resized -p /content/DR_dataset --force
!unzip -o -q /content/DR_dataset/diabetic-retinopathy-resized.zip -d /content/DR_dataset
print("✅ Done")

Dataset URL: https://www.kaggle.com/datasets/tanlikesmath/diabetic-retinopathy-resized
License(s): unknown
100% 7.25G/7.25G [01:13<00:00, 105MB/s] 

✅ Done


### 2.2 Load Labels & Build DataFrame

In [ ]:
CLASS_NAMES  = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative DR']
CLASS_COLORS = ['#27ae60', '#f39c12', '#e67e22', '#e74c3c', '#8e44ad']

IMG_DIR    = Path('/content/DR_dataset/resized_train/resized_train')
LABELS_CSV = Path('/content/DR_dataset/trainLabels.csv')

labels_df = pd.read_csv(LABELS_CSV)
print("Shape   :", labels_df.shape)
print("Columns :", labels_df.columns.tolist())
display(labels_df.head())

Shape   : (35126, 2)
Columns : ['image', 'level']


,image,level
0,10_left,0
1,10_right,0
2,13_left,0
3,13_right,0
4,15_left,1


In [ ]:
records = []
for _, row in tqdm(labels_df.iterrows(), total=len(labels_df), desc='Loading'):
    img_path = IMG_DIR / f"{row['image']}.jpeg"
    if img_path.exists():
        records.append({
            'filepath'  : str(img_path),
            'image'     : row['image'],
            'diagnosis' : int(row['level']),
            'class_name': CLASS_NAMES[int(row['level'])],
        })

df = pd.DataFrame(records)
print(f'✅ Images found   : {len(df):,}')
print(f'   Missing        : {len(labels_df) - len(df):,}')
display(df.head())

df.to_csv('dataset.csv', index=False)
print('💾 Saved: dataset.csv')

Loading:   0%|          | 0/35126 [00:00<?, ?it/s]

✅ Images found   : 35,126
   Missing        : 0


,filepath,image,diagnosis,class_name
0,/content/DR_dataset/resized_train/resized_trai...,10_left,0,No DR
1,/content/DR_dataset/resized_train/resized_trai...,10_right,0,No DR
2,/content/DR_dataset/resized_train/resized_trai...,13_left,0,No DR
3,/content/DR_dataset/resized_train/resized_trai...,13_right,0,No DR
4,/content/DR_dataset/resized_train/resized_trai...,15_left,1,Mild


💾 Saved: dataset.csv
